# 05 Optional Agent 365 extension
## Learning objective
Distinguish prerequisites, read-only resource settings, registry/activity observations and operator attestations.
## Technical level
L300-L400. Expected duration: 20 minutes plus any platform sync wait.
## Prerequisites and setup
Notebook 00; Python 3.12 `.venv`. Bash/PowerShell: `uv sync --frozen --all-extras --dev`. Core Foundry does NOT require Agent 365 licensing. For native features independently verify current licensing, tenant consent/terms, supported prompt-agent type, cloud and preview eligibility, and least-privilege administrative access. Do not grant roles or accept consent from this notebook. Run cell by cell, not unattended Run All; read the [notebook guide](README.md) for consent rules and the production/WAF disclaimer.
## Cost and permissions
First cell is offline. Optional read needs your authorized Foundry account and read permissions; native licensed services may incur separate cost. No automated publish/register/block/delete/logging/consent/governance changes.

## Read-only native evidence, not native onboarding

Agent 365 is a governance and integration surface around an existing agent
runtime. It is not the orchestration loop that decides which function to call.
Microsoft Agent Framework remains an orchestration choice; this application's
function handlers, pre-execution policy and DRY_RUN action boundary remain
application responsibilities. Adding a platform SDK does not transform local
synthetic roles or a demonstration reviewer into authenticated enterprise
authorization.

The current [SDK overview](https://learn.microsoft.com/en-us/microsoft-agent-365/developer/agent-365-sdk?tabs=python)
explicitly distinguishes native platform integration, developer SDK integration,
and connected-platform registry synchronization. Supported native platforms
do not need the Agent 365 SDK merely to appear in inventory. For custom
developer-integrated capabilities, the SDK adds identity, governed Work IQ
tooling and notifications. It does not host, build or orchestrate the agent.
Some capabilities have independent preview, identity and licensing conditions;
inspect current requirements with the responsible administrator.

New observability integrations use the separate
[Microsoft OpenTelemetry Distro](https://learn.microsoft.com/en-us/microsoft-agent-365/developer/microsoft-opentelemetry).
Do not install the deprecated Agent 365 observability SDK. This notebook does
not install either package or reconfigure the repository's existing application
exporter. Older Foundry platform pages may still use earlier SDK terminology
for hosted telemetry; use the current observability guidance when designing a
separately authorized migration. Prompt and hosted agent support must be
assessed independently rather than inferred from a table label.

The experiment here is deliberately smaller than onboarding. First inspect
the adapter's offline prerequisite behavior. Then, only after specific consent,
read existing Foundry account settings through the documented preview option.
Finally compare those facts with separately authorized manual observations of
license, consent, inventory, identity, owner, sponsor and activity. No cell
publishes, registers, assigns roles, accepts tenant terms, changes logging,
blocks an agent, or triggers an attack to generate an alert.

### Step 1 — Establish the adapter's offline limits

**Exact imports:** `agent365.prerequisites`, `agent365.resource_status` and
`agent365.verify_observations`, with `config.Settings.local()`. Inputs are safe
local settings and an empty tuple of observation records. Outputs are
prerequisite/status mappings, not remote Microsoft API responses.

Assertions establish that missing native evidence stays
VERIFICATION_REQUIRED, the local resource report stays UNKNOWN, and empty
observations cannot pass attestation. Printed NOT_IMPLEMENTED and
MANUAL_PORTAL_STEP labels identify adapter boundaries; they are not claims
that the Microsoft product lacks those capabilities. An offline prerequisite
pass would be suspicious here because no native evidence was collected.
These checks need no tenant license or credentials and must never be sold as
successful platform integration.

In [ ]:
from foundry_security_accelerator.agent365 import (
    prerequisites,
    resource_status,
    verify_observations,
)
from foundry_security_accelerator.config import Settings

settings = Settings.local()
checks = prerequisites(settings)
assert any(check["status"] == "VERIFICATION_REQUIRED" for check in checks)
assert resource_status(settings)["status"] == "UNKNOWN"
assert verify_observations((), settings)["status"] == "VERIFICATION_REQUIRED"
print(
    {
        "prerequisites": "VERIFICATION_REQUIRED",
        "registry": "NOT_IMPLEMENTED",
        "activity": "MANUAL_PORTAL_STEP",
    }
)

## Manual portal checklist
With the appropriate owner/administrator, inspect: current license eligibility; existing consent/terms; supported publishing path; actual registry entry; agent identity (not developer credential), owner and sponsor; logging prerequisites; independently correlated activity. Record private observations with timestamps and scope. Review Defender and Purview separately; enabled logging is not an ingested event. Confirm cloud support independently, with no Government parity assumption. See [extension lab](../docs/participant-lab/agent365-extension.md).
If publication, registration, logging or governance changes are needed, stop and obtain independent authorization outside this lab. Do not test Block on shared resources. Observation verification accepts private attestation records but never authenticates the referenced evidence or calls registry APIs.

### Step 2 — Explicitly authorized preview account-setting read

**Exact implementation:** `config.Settings(mode="azure", ...)` supplies
private scope to the already imported `agent365.resource_status`, called
with `preview_api=True`. Input is the owner's configured Foundry account,
not an arbitrary ARM resource. Output is a status dictionary; the cell
prints only status, registry and activity-ingestion labels.

The confirmation approves this documented `2026-03-15-preview` read only.
It does not authorize a PATCH, logging toggle, role assignment, license
change, publishing step, tenant consent or acceptance of terms. The
assertion checks the adapter's deliberate UNKNOWN registry/activity
boundaries, not successful native ingestion. UNKNOWN/PARTIAL native state
is an expected evidence gap, not an assertion that the platform is broken.
If the read is denied, preserve the narrow failure and use offline
prerequisite review. Do not automatically retry with an administrator
credential or infer that licensing caused an authorization error.

In [ ]:
from pathlib import Path

if (
    input("Optional preview account-setting read only. Type READ AUTHORIZED ACCOUNT: ")
    != "READ AUTHORIZED ACCOUNT"
):
    raise PermissionError("Live read not authorized; offline prerequisite review is complete")
ROOT = Path.cwd() if Path("pyproject.toml").exists() else Path.cwd().parent
live = Settings(mode="azure", _env_file=ROOT / ".env")
status = resource_status(live, preview_api=True)
assert status["registry"] == "UNKNOWN" and status["activity_ingestion"] == "UNKNOWN"
print(
    {
        "status": status["status"],
        "registry": status["registry"],
        "activity_ingestion": status["activity_ingestion"],
    }
)

## Expected results and validation
Offline prerequisites stay verification-required. Preview setting reads never prove registry/activity success. Scope/freshness-checked operator observations are OPERATOR_ATTESTATION, not automated service validation. This notebook is never executed in PR CI.
## Security explanation
Foundry is the workload plane; Agent 365 is a separate governance plane. Publishing/registry, resource logging, native activity, Defender and Purview are independent acceptance tasks. Do not invent APIs or fabricate signals. See the [integration guide](../docs/agent365/integration-architecture.md#interpreting-evidence) for blocked/unknown evidence meanings.
## Troubleshooting
Missing license, permission, preview or cloud support: mark the specific blocker if independently observed; otherwise VERIFICATION_REQUIRED. Do not infer the cause or change configuration.
## Cleanup
No mutations performed. Keep observation files private under ignored `.fsa` or `artifacts`, clear outputs, and do not delete agents. [Cleanup](../docs/participant-lab/cleanup.md).
## Source modules
[Agent 365](../src/foundry_security_accelerator/agent365/__init__.py), [availability](../src/foundry_security_accelerator/agent_state.py).

## Separate acceptance tasks and private observations

[Foundry integration](https://learn.microsoft.com/en-us/azure/foundry/agents/concepts/agent-365-integration)
distinguishes publication/registry synchronization from workload creation.
[Data collection guidance](https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/configure-agent-365-data-collection)
describes account-level `a365LoggingEnabled` and read-only `a365Status`.
License, accepted tenant terms and eligible collection still need independent
confirmation. A boolean setting is not an ingested activity event. The account
setting covers contained projects/prompt agents, so toggling it is not a
harmless per-notebook preference and is outside this lab.

The adapter's `Observation` input is an array of records with capability,
tenant, agent, observer, observed_at, evidence_url, status and notes. Required
capabilities are license, consent, registry, identity, owner, sponsor and
activity. Verification checks configured tenant/agent scope, duplicates,
completeness and freshness within 24 hours. It does not authenticate the
observer or fetch/authenticate the evidence URL. Use real authorized private
observations; do not manufacture records to satisfy the schema.

| Evidence label | What it proves | What it cannot prove |
|---|---|---|
| UNKNOWN / VERIFICATION_REQUIRED | Evidence is absent or not established | A specific licensing or service failure |
| READ_ONLY_VERIFICATION | Scoped setting was read | Registry/activity existence |
| MANUAL_PORTAL_STEP | A native observation is still required | Automated validation |
| OPERATOR_ATTESTATION PASS | Supplied records meet scope/time/completeness checks | API-verified native success |
| Blocker independently observed | That scoped blocker was seen at that time | Universal cloud/product unavailability |

## Questions and read-only exercises

Explain the difference between the developer's DefaultAzureCredential caller,
an agent identity and a synthetic application role. Which independent evidence
would show an agent is published rather than merely created? Why does an
inventory entry not prove activity ingestion? Identify where an owner/sponsor
observation belongs and why it cannot approve a business write. Review existing
Defender and Purview signals only if specifically authorized; absence of a
finding is not evidence that no request occurred. Do not generate attacks,
block a shared agent or alter policies for a screenshot.

## Least privilege, cost, troubleshooting and cleanup

Use only scoped read permission and separately authorized portal access.
A license purchase, publication, administrative consent, RBAC assignment,
logging change or preview enrollment requires its own independent review and
authorization outside this notebook. Ambiguous permission to “check native
integration” is not permission to mutate it. Sovereign-cloud and hosted-agent
support stay VERIFICATION_REQUIRED unless specifically established.

Keep private observations under ignored storage with the approved retention
policy; IDs and portal captures can be sensitive without containing secrets.
Clear outputs, discard loaded settings by restarting the kernel, and do not
delete or retire agents here. No cloud resources were created by this notebook.
If evidence is absent, continue with the explicitly LOCAL fallback rather than
inventing an API or implying native success.

## Read saved LIVE evidence without substituting local fixtures

In a separately authorized private terminal, the existing CLI can inspect
actual retained live evidence:

```text
uv run fsa --mode azure audit verify
uv run fsa --mode azure trace list
uv run fsa --mode azure trace show S4
uv run fsa azure agent-status --profile presentation
```

Azure-mode audit verification reads the latest LIVE session, and Azure-mode
trace list/show reads saved live scenes/spans. `S4` selects a saved scene;
if it is absent, retain the absence rather than substituting a local
fixture or fabricated trace. PRESENTATION reports store the local span
reference in `trace_path`. Saved spans are application evidence, not proof
of exporter delivery, workspace ingestion or native product activity.

For existing Agent 365 read-only commands, add `--profile presentation`
when that is the intended dedicated agent rather than EQUIPMENT, for example
`uv run fsa agent365 status --profile presentation`.
Profile selection does not authorize preview reads, native changes,
publication, consent or role assignments. Review the existing preview
prompt independently. Account enablement, registry, activity, Defender and
Purview remain separate evidence scopes; no environment result is embedded
in this notebook.

## Current platform references and claim boundaries

References checked on 2026-11-12; these are reading links, not remote code to execute.
[Foundry function calling](https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/tools/function-calling)
documents the model-request/client-execution loop. The PRESENTATION profile attaches
four function definitions directly; no Toolbox is required. The model requests
`search_policies`, `get_policy_section`, `get_ticket_status`, or
`create_records_request`; application policy checks each proposed execution first.
Function selection is not authorization. Only the approved synthetic Azure Search
corpus is eligible, and a records request enters a local persistent approval queue.
Even in Azure mode, supplied action receipts are **DRY_RUN**, and the identity label
is **SYNTHETIC_ROLE_LABELS_NOT_ENTRA_OBO**, not authenticated enterprise approval.
The existing `presentation_runtime.list_approvals(...)` and
`presentation_runtime.approve_request(...)` operate on the shared live/local
persistent queue. Inspect their documented inputs privately before use; these
notebooks never automatically approve requests. Synthetic reviewer labels are
not authenticated humans. Function schemas may be visible in the Foundry portal,
but the playground does not execute this application's local function handlers.
Use the authorized application client loop, not a portal chat, to test execution.

Native scope is **read-only evidence checks and detailed portal/SDK guidance**.
There is no authorization here for publishing, native configuration, tenant
consent, RBAC, licensing or governance mutations. Telemetry export configuration
requires separate reviewed approval. Added live invocation cells require an
independent default-off authorization for already-configured Monitor/OTLP egress;
invocation consent alone does not permit export. No cell configures an exporter.
The six-scene initial verification budget is at most 18 model calls across the
entire initial verification, not 18 per notebook. Choose the cells you need and
subtract any prior calls; do not run all optional examples within that budget.

[Agent 365 SDK overview](https://learn.microsoft.com/en-us/microsoft-agent-365/developer/agent-365-sdk?tabs=python)
states that supported platforms use native integration without adding the SDK.
The SDK extends an existing runtime with identity, governed Work IQ tooling, and
notifications; it does not build, host, or orchestrate agents.
[Microsoft OpenTelemetry Distro](https://learn.microsoft.com/en-us/microsoft-agent-365/developer/microsoft-opentelemetry)
is the separate current observability integration. Do **not** add the deprecated
Agent 365 observability SDK. These references describe platform choices, not
packages installed or tenant capabilities validated by this notebook.

Official source repositories for further study:
[Microsoft Agent Framework](https://github.com/microsoft/agent-framework)
(agent orchestration), [Azure AI Projects SDK](https://github.com/Azure/azure-sdk-for-python/tree/main/sdk/ai/azure-ai-projects)
(Foundry client contracts), [Agent 365 Python SDK](https://github.com/microsoft/Agent365-python)
(selected developer integrations), and
[Microsoft OpenTelemetry Python Distro](https://github.com/microsoft/opentelemetry-distro-python)
(current observability). Follow this repository's pinned environment rather than
installing arbitrary versions from sample pages. No upstream code is copied here.